<a href="https://colab.research.google.com/github/robin8a/sse_etl/blob/main/GLV_ETL_Providers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup Environment

## ETL to load items from providers

Help to create a script  to extract, transfrom and load (ETL) data from a google drive google sheet file, another google sheet as a template, and output file also a google sheet file.  

## Help to update the script with:

- Also in drive params.json with the following structure

```json
{
  "xlsx_file_provider": "",
  "xlsx_file_provider_sheets": [""],
  "xlsx_file_provider_row_to_start": "",
  "xlsx_file_provider_has_subcategory": "",  
  "xlsx_file_template": "",
  "xlsx_file_output": "",

}
```

- The parameter "xlsx_file_provider_row_to_start" is the row that contains the columuns to match with
- The parameter "xlsx_file_provider_has_subcategory" if is true, means there are row that has subcategory value in the row not the N columns as is in "xlsx_file_provider_row_to_start"
- Allow me to match "xlsx_file_provider" columns with the "xlsx_file_template" columnms, also allow me to check if the column does exist an added to end with "NOT_MATH_<column_name>
- The source file also includes and embed "image" column that needs to be extracted and upload to AWS S3 bucket.





In [2]:
!pip install boto3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 7.7 MB/s eta 0:00:00


In [10]:
import json
import pandas as pd
import gspread
from google.colab import auth
import boto3
from io import StringIO
import re
import os
import requests # For downloading images
import mimetypes # To guess image file types
from google.auth import default # Import default

# Install required libraries
!pip install --upgrade gspread pandas boto3 --quiet

## Google Drive Authentication

To write to Google Sheets, you need to authenticate this Colab notebook with your Google account. Run the following cell and follow the instructions to grant access.

## AWS S3 Configuration

To upload images to AWS S3, you will need to provide your AWS credentials. It's recommended to store these as Colab secrets for security. Click on the '🔑' icon in the left sidebar, then add your `AWS_ACCESS_KEY_ID` and `AWS_SECRET_ACCESS_KEY`.

You will also need to specify your AWS region and S3 bucket name.

In [45]:
# Import the Colab userdata library
from google.colab import userdata

# Retrieve AWS credentials from Colab secrets
AWS_ACCESS_KEY_ID = userdata.get('sse_aws_access_key_id')
AWS_SECRET_ACCESS_KEY = userdata.get('sse_aws_secret_access_key')
AWS_REGION = 'your-aws-region' # e.g., 'us-east-1', update this
AWS_S3_BUCKET_NAME = 'your-s3-bucket-name' # update this with your S3 bucket name

# Initialize S3 client
s3_client = boto3.client(
    's3',
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    region_name=AWS_REGION
)

## Configuration Parameters (params.json)

Below is the structure for your `params.json` file. Please update the placeholder values with your actual Google Sheet file IDs or names, and other relevant parameters. You can run this cell to create a `params.json` file in your Colab environment or manually upload one to `/content/params.json`.

In [46]:
params_content = {
  "xlsx_file_provider": "/content/ListadoGVS(13.04.26).xlsx", # Path to the provider Excel file
  "xlsx_file_provider_sheets": ["OUTLET"], # Array of sheet names to process from the provider Excel, e.g., ["Sheet1", "Sheet2"]
  "xlsx_file_provider_row_to_start": 4, # Row number (1-indexed) containing column headers in provider Excel
  "txt_file_match": "/content/match.txt", # Path to the text file for column matching
  "csv_file_output": "/content/output.csv", # Path for the output CSV file
  "s3_folder_path": "ssebeapief4a7d425930451b867d315da2ab24c9cba30-dev/providers_data/" # Folder path in S3 bucket for images
}

# Save this to a params.json file
with open('params.json', 'w') as f:
    json.dump(params_content, f, indent=2)

print("params.json created. Please update its content in the file browser or directly in this cell.")

params.json created. Please update its content in the file browser or directly in this cell.


## Load Parameters

Run this cell to load the parameters from `params.json`.

In [40]:
with open('params.json', 'r') as f:
    params = json.load(f)

# Display loaded parameters for verification
print(json.dumps(params, indent=2))

{
  "xlsx_file_provider": "/content/ListadoGVS(13.04.26).xlsx",
  "xlsx_file_provider_sheets": [
    "OUTLET"
  ],
  "xlsx_file_provider_row_to_start": 4,
  "xlsx_file_provider_has_subcategory": true,
  "txt_file_match": "/content/match.txt",
  "csv_file_output": "/content/output.csv",
  "s3_folder_path": "ssebeapief4a7d425930451b867d315da2ab24c9cba30-dev/providers_data/"
}


## Load Parameters

Run this cell to load the parameters from `params.json`.

In [41]:
with open('params.json', 'r') as f:
    params = json.load(f)

# Display loaded parameters for verification
print(json.dumps(params, indent=2))

{
  "xlsx_file_provider": "/content/ListadoGVS(13.04.26).xlsx",
  "xlsx_file_provider_sheets": [
    "OUTLET"
  ],
  "xlsx_file_provider_row_to_start": 4,
  "xlsx_file_provider_has_subcategory": true,
  "txt_file_match": "/content/match.txt",
  "csv_file_output": "/content/output.csv",
  "s3_folder_path": "ssebeapief4a7d425930451b867d315da2ab24c9cba30-dev/providers_data/"
}


In [42]:
print(json.dumps(params, indent=2))

{
  "xlsx_file_provider": "/content/ListadoGVS(13.04.26).xlsx",
  "xlsx_file_provider_sheets": [
    "OUTLET"
  ],
  "xlsx_file_provider_row_to_start": 4,
  "xlsx_file_provider_has_subcategory": true,
  "txt_file_match": "/content/match.txt",
  "csv_file_output": "/content/output.csv",
  "s3_folder_path": "ssebeapief4a7d425930451b867d315da2ab24c9cba30-dev/providers_data/"
}


## ETL Script Outline

Here's a plan to build the ETL script:

1.  **Helper Functions**: Define functions for reading/writing Excel/CSV and uploading to S3.
2.  **Extract Data**: Read the provider and template Excel files based on `params.json`.
3.  **Process Provider Sheets**: Iterate through each sheet specified in `xlsx_file_provider_sheets`.
    *   Handle `xlsx_file_provider_row_to_start` to identify headers.
    *   Implement column matching logic between provider data and template columns.
    *   Identify and mark unmatched columns.
    *   Process `xlsx_file_provider_has_subcategory` to correctly extract data.
4.  **Image Extraction and Upload**: Identify the 'image' column, download images, and upload them to AWS S3, replacing URLs in the data.
5.  **Load Data**: Write the transformed data to the output CSV file.

In [47]:
with open('params.json', 'r') as f:
    params = json.load(f)

# Load template columns and their mappings from match.txt
column_mapping = {}
template_columns = []
with open(params['txt_file_match'], 'r') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        parts = line.split('|')
        template_col_name = parts[0].strip().lower()
        provider_col_name = parts[1].strip().lower() if len(parts) > 1 else ''
        column_mapping[template_col_name] = provider_col_name
        template_columns.append(template_col_name)

# Initialize an empty DataFrame to store all processed data
all_processed_data = pd.DataFrame()

# Determine sheets to process
provider_sheets = params['xlsx_file_provider_sheets']
if not provider_sheets or (len(provider_sheets) == 1 and provider_sheets[0] == ""):
    # If empty or [''], read all sheets from the provider Excel file
    excel_file = pd.ExcelFile(params['xlsx_file_provider'])
    actual_provider_sheets = excel_file.sheet_names
else:
    actual_provider_sheets = provider_sheets

for sheet_name in actual_provider_sheets:
    print(f"Processing sheet: {sheet_name}")
    try:
        # Extract data from provider sheet
        provider_df = read_excel_to_dataframe(
            params['xlsx_file_provider'],
            sheet_name,
            params['xlsx_file_provider_row_to_start']
        )

        # --- Remove duplicated rows ---
        initial_rows = len(provider_df)
        provider_df.drop_duplicates(inplace=True)
        if len(provider_df) < initial_rows:
            print(f"Removed {initial_rows - len(provider_df)} duplicate rows from sheet '{sheet_name}'.")

        print(f"DataFrame after deduplication for sheet '{sheet_name}':")
        display(provider_df.head())
        # -----------------------------

        # Normalize provider column names for matching
        provider_df.columns = [col.strip().lower() for col in provider_df.columns]

        # Process column matching and add missing template columns
        processed_df = pd.DataFrame()
        used_provider_cols = set()

        for template_col in template_columns:
            mapped_provider_col = column_mapping.get(template_col)
            if mapped_provider_col and mapped_provider_col in provider_df.columns:
                processed_df[template_col] = provider_df[mapped_provider_col]
                used_provider_cols.add(mapped_provider_col)
            else:
                processed_df[template_col] = pd.NA # Add missing template columns or if mapped column not found

        # Identify and add unmatched provider columns
        unmatched_provider_cols_list = []
        for provider_col in provider_df.columns:
            # Exclude original category/subcategory if they are added by read_excel_to_dataframe and then handled
            # And exclude columns that were explicitly used in the mapping
            if provider_col not in used_provider_cols and \
               provider_col not in template_columns and \
               provider_col != 'category' and provider_col != 'subcategory' and \
               provider_col not in column_mapping.values(): # Also exclude columns that were *intended* to be mapped but might not exist
                processed_df[f"NOT_MATCH_{provider_col}"] = provider_df[provider_col]
                unmatched_provider_cols_list.append(provider_col)

        if unmatched_provider_cols_list:
            print(f"Unmatched columns in sheet '{sheet_name}' from provider data: {', '.join(unmatched_provider_cols_list)}")

        # Ensure 'Category' column (from sheet name) is present
        if 'category' in template_columns and 'category' not in processed_df.columns:
            processed_df['category'] = sheet_name
        elif 'category' in processed_df.columns and processed_df['category'].isnull().all():
             processed_df['category'] = sheet_name

        # --- Image Extraction and Upload ---
        # Assuming an 'image' column exists in the processed_df
        # If the template defines an 'image' column, we will process it.
        if 'image' in processed_df.columns:
            print(f"Uploading images for sheet: {sheet_name}")
            # Ensure S3 client and bucket are configured
            if 'AWS_ACCESS_KEY_ID' in globals() and AWS_ACCESS_KEY_ID and \
               'AWS_SECRET_ACCESS_KEY' in globals() and AWS_SECRET_ACCESS_KEY and \
               'AWS_S3_BUCKET_NAME' in globals() and AWS_S3_BUCKET_NAME and \
               's3_client' in globals() and s3_client:
                processed_df['image'] = processed_df['image'].apply(
                    lambda img_url: upload_image_to_s3(
                        s3_client,
                        AWS_S3_BUCKET_NAME,
                        params['s3_folder_path'],
                        img_url
                    ) if pd.notna(img_url) else img_url
                )
            else:
                print("AWS S3 credentials or bucket not fully configured. Skipping image upload.")
                print("Please set AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY, AWS_REGION, AWS_S3_BUCKET_NAME.")

        # Append to the combined DataFrame
        all_processed_data = pd.concat([all_processed_data, processed_df], ignore_index=True)

    except Exception as e:
        print(f"Error processing sheet {sheet_name}: {e}")
        continue # Continue to the next sheet even if one fails

# --- Load Data ---
# Write the final combined DataFrame to CSV
if not all_processed_data.empty:
    output_file_path = params['csv_file_output']
    if not output_file_path:
        print("Warning: 'csv_file_output' is not specified in params.json. Skipping CSV output.")
    else:
        # Convert all columns to string type before writing to CSV to avoid dtype issues
        # and ensure consistent output, especially for mixed types or NA values.
        all_processed_data = all_processed_data.astype(str)
        write_dataframe_to_csv(all_processed_data, output_file_path)
else:
    print("No data processed to write to output file.")

print("ETL process completed.")

Processing sheet: OUTLET
Removed 1 duplicate rows from sheet 'OUTLET'.
DataFrame after deduplication for sheet 'OUTLET':


,,Codigo,-IP NVR SERIE 7600,Precio,Precio IVA,Descuento,Precio Oferta IVA Incluido,Vigencia Oferta,Grupo,Unnamed: 9,Category
0,NaN,GVIP7608E2,NVR 8 CH HDMI/VGA SIN FRONTAL 2 SATAS 1 RJ-45...,88015,104737,NaN,NaN,NaN,NaN,NaN,OUTLET
1,,Codigo,-HDTVI DVR SERIE 7100 720P,Precio,Precio IVA,Descuento,Precio Oferta IVA Incluido,Vigencia Oferta,Grupo,NaN,OUTLET
2,NaN,GV7108GF1,** Existencia Única **\nMINI DVR 8CH TURBO HD...,70435,83817,NaN,NaN,NaN,NaN,NaN,OUTLET
3,HIKVISION,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,OUTLET
4,,Codigo,-4 EN 1 BALA VARIFOCAL 720P,Precio,Precio IVA,Descuento,Precio Oferta IVA Incluido,Vigencia Oferta,Grupo,NaN,OUTLET


Unmatched columns in sheet 'OUTLET' from provider data: codigo, -ip nvr serie 7600, precio, precio iva, descuento, precio oferta iva incluido, vigencia oferta, grupo, unnamed: 9
Uploading images for sheet: OUTLET
Successfully wrote data to CSV file: /content/output.csv
ETL process completed.


## Helper Functions

In [48]:
def get_sheet_by_name_or_id(gc, title_or_id):
    """Gets a Google Sheet by its title or ID."""
    try:
        # Try as ID first
        sheet = gc.open_by_key(title_or_id)
    except gspread.exceptions.SpreadsheetNotFound:
        try:
            # Then try as title
            sheet = gc.open(title_or_id)
        except gspread.exceptions.SpreadsheetNotFound:
            raise ValueError(f"Google Sheet not found with ID or name: {title_or_id}")
    return sheet

def read_excel_to_dataframe(
    file_path,
    sheet_name,
    header_row_index
):
    """Reads a specific worksheet from a local Excel file into a Pandas DataFrame."""
    try:
        # The header_row_index is 1-indexed, so convert to 0-indexed for pandas
        df = pd.read_excel(file_path, sheet_name=sheet_name, header=header_row_index - 1)
    except Exception as e:
        raise ValueError(f"Error reading Excel file {file_path}, sheet {sheet_name}: {e}")

    # Remove empty rows after reading
    df.replace('', pd.NA, inplace=True)
    df.dropna(how='all', inplace=True)

    # Assign worksheet_name to a 'Category' column, as requested.
    df['Category'] = sheet_name

    return df

def upload_image_to_s3(s3_client, bucket_name, s3_folder_path, image_url):
    """Downloads an image from a URL and uploads it to S3."""
    if not image_url or not image_url.startswith(('http://', 'https://')):
        return image_url # Return original if not a valid URL

    try:
        response = requests.get(image_url, stream=True, timeout=10)
        response.raise_for_status() # Raise an exception for bad status codes

        # Guess file extension from content type or URL
        content_type = response.headers.get('content-type')
        ext = mimetypes.guess_extension(content_type) if content_type else None
        if not ext and '.' in image_url.split('?')[-2]:
            ext = '.' + image_url.split('?')[-2].split('.')[-1]
        if not ext:
            ext = '.jpg'

        # Extract filename from URL or generate one
        image_name = os.path.basename(image_url.split('?')[0])
        if not image_name or '.' not in image_name:
             # Generate a simple name if not present or lacks extension
            image_name = f"image_{hash(image_url)}{ext}"
        elif not image_name.endswith(ext):
            # Ensure the extension is correct
            image_name = f"{os.path.splitext(image_name)[0]}{ext}"

        s3_key = f"{s3_folder_path}/{image_name}"

        s3_client.upload_fileobj(response.raw, bucket_name, s3_key)
        s3_public_url = f"https://{bucket_name}.s3.{s3_client.meta.region_name}.amazonaws.com/{s3_key}"
        return s3_public_url

    except requests.exceptions.RequestException as e:
        print(f"Error downloading {image_url}: {e}")
        return image_url # Return original URL on error
    except Exception as e:
        print(f"Error uploading {image_url} to S3: {e}")
        return image_url # Return original URL on error

def write_dataframe_to_csv(df, output_path):
    """Writes a Pandas DataFrame to a CSV file."""
    try:
        df.to_csv(output_path, index=False)
        print(f"Successfully wrote data to CSV file: {output_path}")
    except Exception as e:
        raise ValueError(f"Error writing DataFrame to CSV {output_path}: {e}")

def write_dataframe_to_google_sheet(gc, df, sheet_title_or_id, worksheet_name):
    """This function is no longer used as output is now CSV. Writes a Pandas DataFrame to a specified worksheet in a Google Sheet."""
    # Keeping the function definition but marking it as unused.
    # The ETL will now call write_dataframe_to_csv instead.
    print("Warning: write_dataframe_to_google_sheet is deprecated and will not be called for CSV output.")
    spreadsheet = get_sheet_by_name_or_id(gc, sheet_title_or_id)
    try:
        worksheet = spreadsheet.worksheet(worksheet_name)
    except gspread.exceptions.WorksheetNotFound:
        worksheet = spreadsheet.add_worksheet(title=worksheet_name, rows=df.shape[0]+1, cols=df.shape[1])

    # Clear existing content
    worksheet.clear()

    # Convert DataFrame to a list of lists, including header
    data_to_write = [df.columns.values.tolist()] + df.values.tolist()

    # Update the worksheet
    worksheet.update(data_to_write)
    print(f"Successfully wrote data to Google Sheet '{sheet_title_or_id}', worksheet '{worksheet_name}'.")

### Revisar el Archivo `output.csv`

Cargamos el archivo CSV generado para inspeccionar los mapeos de columnas y los datos resultantes.

In [49]:
output_df = pd.read_csv(params['csv_file_output'])
print(f"Filas en el output.csv: {len(output_df)}")
display(output_df.head())

Filas en el output.csv: 76


,item_category,general_profit_percentage,name_to_show,own_name,own_inventory_code,provider_name,provider_inventory_code,description,unit_of_measure,image,...,own_value,NOT_MATCH_codigo,NOT_MATCH_-ip nvr serie 7600,NOT_MATCH_precio,NOT_MATCH_precio iva,NOT_MATCH_descuento,NOT_MATCH_precio oferta iva incluido,NOT_MATCH_vigencia oferta,NOT_MATCH_grupo,NOT_MATCH_unnamed: 9
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,GVIP7608E2,NVR 8 CH HDMI/VGA SIN FRONTAL 2 SATAS 1 RJ-45...,88015,104737,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,Codigo,-HDTVI DVR SERIE 7100 720P,Precio,Precio IVA,Descuento,Precio Oferta IVA Incluido,Vigencia Oferta,Grupo,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,GV7108GF1,** Existencia Única **\nMINI DVR 8CH TURBO HD...,70435,83817,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,Codigo,-4 EN 1 BALA VARIFOCAL 720P,Precio,Precio IVA,Descuento,Precio Oferta IVA Incluido,Vigencia Oferta,Grupo,NaN
